# Notebook 1: Distributions & Randomness
**Stats for Options Trading — Problem Set 1**

This notebook is structured as: **concept → working example → your challenge**.
Don't skip the challenges. That's where the learning happens.

By the end of this notebook you will:
- Understand what a probability distribution actually *is*
- Know why the normal distribution is central to options pricing (and where it breaks)
- Have hands-on intuition for expected value, variance, and standard deviation
- See your first connection to implied volatility

---

## Setup — Run This First

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats

# Makes plots look clean
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

# Fix random seed so your results are reproducible
np.random.seed(42)

print('Setup complete.')

---
## Part 1: What Is a Distribution?

A **distribution** is just an answer to the question: *"Of all the possible outcomes, how likely is each one?"*

The simplest example: a coin flip. Two outcomes, each with 50% probability.
Flip it 1000 times and count the results — that count *is* the distribution.

Run the cell below, then read what it's telling you.

In [ ]:
# Simulate 1000 coin flips (0 = tails, 1 = heads)
n_flips = 1000
flips = np.random.randint(0, 2, size=n_flips)

n_heads = flips.sum()
n_tails = n_flips - n_heads

print(f'Heads: {n_heads} ({n_heads/n_flips:.1%})')
print(f'Tails: {n_tails} ({n_tails/n_flips:.1%})')

plt.bar(['Tails', 'Heads'], [n_tails, n_heads], color=['steelblue', 'coral'])
plt.axhline(y=500, color='black', linestyle='--', alpha=0.4, label='Expected (50%)')
plt.title('1000 Coin Flips')
plt.ylabel('Count')
plt.legend()
plt.show()

### 🎯 Challenge 1.1
Change `n_flips` to 10, then to 10,000,000. 
- What happens to the % of heads as flips increase? 
- This is the **Law of Large Numbers** — write a one-line comment in the cell explaining it in your own words.

In [ ]:
# YOUR WORK HERE
# Try n_flips = 10, then 1000, then 10_000_000
n_flips = 10  # <-- change this

flips = np.random.randint(0, 2, size=n_flips)
n_heads = flips.sum()
print(f'Heads: {n_heads/n_flips:.1%}')

# Law of Large Numbers means: (write your explanation here)

---
## Part 2: The Normal Distribution

The **normal distribution** (bell curve) is everywhere in statistics — and it's the foundation of Black-Scholes options pricing.

It's defined by exactly two numbers:
- **Mean (μ)** — the center. Where outcomes cluster.
- **Standard deviation (σ)** — the spread. How wide the bell is.

In options, σ *is* volatility. When a stock has 20% implied vol, you're saying:
> "The market prices this stock as if annual returns follow a normal distribution with σ = 20%."

Run the cell and pay attention to what changing σ does to the shape.

In [ ]:
x = np.linspace(-5, 5, 1000)

# Plot three normal distributions with different spreads
for sigma, color, label in [(0.5, 'steelblue', 'σ = 0.5 (low vol)'),
                             (1.0, 'coral',     'σ = 1.0 (medium vol)'),
                             (2.0, 'green',     'σ = 2.0 (high vol)')]:
    y = stats.norm.pdf(x, loc=0, scale=sigma)
    plt.plot(x, y, label=label, color=color, linewidth=2)

plt.title('Normal Distributions — Same Mean, Different Volatility')
plt.xlabel('Return')
plt.ylabel('Probability Density')
plt.legend()
plt.show()

print('Notice: wider = more uncertainty = higher vol = more expensive options')

### 🎯 Challenge 2.1 — The 68-95-99.7 Rule

For a normal distribution:
- **68%** of outcomes fall within 1σ of the mean
- **95%** fall within 2σ
- **99.7%** fall within 3σ

This is *critical* for options. A 1-standard-deviation move in the underlying corresponds roughly to a 16-delta option expiring in-the-money.

Verify this rule by simulation below:

In [ ]:
# Simulate 100,000 draws from a normal distribution
mu = 0
sigma = 1
samples = np.random.normal(loc=mu, scale=sigma, size=100_000)

# What % falls within 1 standard deviation?
within_1sd = np.mean(np.abs(samples - mu) <= 1 * sigma)
within_2sd = np.mean(np.abs(samples - mu) <= 2 * sigma)
within_3sd = np.mean(np.abs(samples - mu) <= 3 * sigma)

print(f'Within 1σ: {within_1sd:.1%}  (theory: 68.3%)')
print(f'Within 2σ: {within_2sd:.1%}  (theory: 95.4%)')
print(f'Within 3σ: {within_3sd:.1%}  (theory: 99.7%)')

# CHALLENGE: Change sigma to 0.2 (think: 20% annual vol)
# What % of outcomes are within 0.2 of the mean?
# Does the 68-95-99.7 rule still hold? Why?

---
## Part 3: Expected Value and Variance

**Expected Value (EV)** = the average outcome if you repeated the bet forever.
**Variance** = how spread out the outcomes are. Std dev = √variance.

These two numbers *summarize* a distribution. Everything in options pricing builds on them.

Real example: a simple options trade as a bet.

In [ ]:
# Simple bet: You buy a call option for $2 premium
# Scenario A (60% prob): Option expires worthless -> P&L = -$2
# Scenario B (40% prob): Option expires worth $8 -> P&L = +$6 (8 - 2 premium)

outcomes      = np.array([-2, 6])      # P&L in each scenario
probabilities = np.array([0.60, 0.40]) # must sum to 1.0

ev       = np.sum(outcomes * probabilities)
variance = np.sum(probabilities * (outcomes - ev)**2)
std_dev  = np.sqrt(variance)

print(f'Expected Value : ${ev:.2f}')
print(f'Variance       : {variance:.2f}')
print(f'Std Deviation  : ${std_dev:.2f}')
print()
print('Interpretation:')
print(f'  On average, this trade makes ${ev:.2f} per contract.')
print(f'  Outcomes typically deviate from that by ~${std_dev:.2f}.')

### 🎯 Challenge 3.1 — Break-Even Probability

What probability of the option paying off makes this a **zero expected value** trade?

Hint: set EV = 0 and solve for p. Then verify it with the code below.

In [ ]:
# YOUR WORK HERE
# Find the breakeven probability where EV = 0
# Outcomes are still [-2, +6]
# Solve: p_win * 6 + (1 - p_win) * (-2) = 0

# p_win = ???  (solve it on paper first, then verify here)
p_win = 0.0  # <-- fill this in
p_lose = 1 - p_win

ev_check = p_win * 6 + p_lose * (-2)
print(f'EV at p_win={p_win:.0%}: ${ev_check:.4f}')
print('Should be $0.00 if you solved it correctly.')

# BONUS: This breakeven probability is exactly what a call option's delta approximates.
# A 25-delta call = ~25% chance of expiring ITM at the moment of pricing.
# Does that change how you think about buying vs selling options?

---
## Part 4: Fat Tails — Where the Normal Distribution Breaks

Black-Scholes assumes stock returns are normally distributed. 
**Real markets have fat tails** — extreme moves happen far more often than the normal distribution predicts.

This is why:
- OTM options are often "mispriced" relative to BS
- Vol surfaces have skew and kurtosis
- Tail-risk hedging is a real business

Let's see the difference visually.

In [ ]:
n = 100_000

# Normal distribution
normal_returns = np.random.normal(loc=0, scale=1, size=n)

# Student's t-distribution with low degrees of freedom = fat tails
# Lower df = fatter tails. Markets often behave like df=3 to df=5
fat_tail_returns = np.random.standard_t(df=4, size=n)

# Plot both
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, data, title, color in [
    (axes[0], normal_returns,   'Normal Distribution',       'steelblue'),
    (axes[1], fat_tail_returns, 'Fat Tails (t, df=4)',       'coral')
]:
    ax.hist(data, bins=200, density=True, alpha=0.7, color=color)
    ax.set_xlim(-8, 8)
    ax.set_title(title)
    ax.set_xlabel('Return (σ)')
    ax.set_ylabel('Density')

plt.suptitle('Normal vs Fat Tails — Look at the Extremes', fontsize=13)
plt.tight_layout()
plt.show()

# Count extreme events (>3σ moves)
normal_extremes   = np.mean(np.abs(normal_returns) > 3)
fattail_extremes  = np.mean(np.abs(fat_tail_returns) > 3)
print(f'3σ+ moves in normal dist : {normal_extremes:.2%}')
print(f'3σ+ moves in fat-tail    : {fattail_extremes:.2%}')
print(f'Fat tails have {fattail_extremes/normal_extremes:.1f}x more extreme events')

### 🎯 Challenge 4.1
Change `df` in the t-distribution from 4 to 30. What happens to the tail behavior?

Then look up: **what does `df → ∞` converge to?** (Hint: it becomes the normal distribution.)

Write your observation as a comment.

In [ ]:
# YOUR WORK HERE
# Experiment with different df values: 2, 4, 10, 30, 100
df = 4  # <-- change this

fat_tail_returns = np.random.standard_t(df=df, size=100_000)
extremes = np.mean(np.abs(fat_tail_returns) > 3)
print(f'df={df}: {extremes:.2%} of moves exceed 3σ')

# As df increases, the distribution looks more like: ___________
# This matters for trading because: ___________

---
## Part 5: Simulating Stock Returns

Now let's connect everything to actual markets.

A stock's daily return can be modeled as a draw from a normal distribution:
- **Mean (μ)** ≈ small daily drift (often ~0 for modeling purposes)
- **Std dev (σ)** = daily vol = annualized vol / √252

If SPY has 15% annual vol, daily vol ≈ 15% / √252 ≈ **0.94% per day**.

In [ ]:
# Simulate 1 year of daily SPY returns
annual_vol = 0.15          # 15% implied vol
daily_vol  = annual_vol / np.sqrt(252)
n_days     = 252
S0         = 100           # starting price

daily_returns = np.random.normal(loc=0, scale=daily_vol, size=n_days)
price_path    = S0 * np.exp(np.cumsum(daily_returns))  # geometric returns

plt.plot(price_path, color='steelblue', linewidth=1.5)
plt.axhline(y=S0, color='black', linestyle='--', alpha=0.4, label='Start price')
plt.title(f'Simulated Stock Path (Annual Vol = {annual_vol:.0%})')
plt.xlabel('Trading Days')
plt.ylabel('Price')
plt.legend()
plt.show()

print(f'Start:     ${S0:.2f}')
print(f'End:       ${price_path[-1]:.2f}')
print(f'Daily vol: {daily_vol:.3%}')

### 🎯 Challenge 5.1 — Monte Carlo (Your First One)

Run 1,000 simulated price paths and answer: 
1. What % of paths end above the starting price of $100?
2. What % end below $90? (Think: how far OTM is an 90-strike put?)

This is basically a Monte Carlo pricer — the foundation of how firms price exotic options.

In [ ]:
# YOUR WORK HERE — Monte Carlo simulation
n_simulations = 1000
annual_vol    = 0.15
daily_vol     = annual_vol / np.sqrt(252)
n_days        = 252
S0            = 100

final_prices = []

for _ in range(n_simulations):
    daily_returns = np.random.normal(loc=0, scale=daily_vol, size=n_days)
    final_price   = S0 * np.exp(np.sum(daily_returns))
    final_prices.append(final_price)

final_prices = np.array(final_prices)

# Plot the distribution of final prices
plt.hist(final_prices, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
plt.axvline(x=S0,  color='black',  linestyle='--', label='Start ($100)')
plt.axvline(x=90,  color='coral',  linestyle='--', label='$90 strike')
plt.title('Distribution of Final Prices After 1 Year (1,000 Simulations)')
plt.xlabel('Final Price')
plt.ylabel('Count')
plt.legend()
plt.show()

# Answer the questions:
pct_above_100 = np.mean(final_prices > 100)
pct_below_90  = np.mean(final_prices < 90)

print(f'% paths ending above $100 : {pct_above_100:.1%}')
print(f'% paths ending below $90  : {pct_below_90:.1%}')
print()
print('BONUS CHALLENGE:')
print('Change annual_vol to 0.30 (30%). How does that change both percentages?')
print('Why does higher vol INCREASE the chance of finishing above AND below?')

---
## Recap — What You Just Learned

| Concept | What it is | Why it matters for options |
|---|---|---|
| Distribution | Map of all possible outcomes + their probabilities | Options price the full distribution of a stock |
| Normal distribution | Bell curve defined by μ and σ | Foundation of Black-Scholes |
| Standard deviation | Spread of outcomes | σ = volatility. Wider = more expensive options |
| Expected value | Average outcome across infinite trials | Every options trade has an EV. Know yours. |
| Fat tails | Extreme moves happen more than normal dist predicts | Why vol skew exists. Why OTM puts are expensive. |
| Monte Carlo | Simulate thousands of paths, count outcomes | How complex options get priced |

---
## Next Notebook

**Notebook 2: Bayesian Thinking** — How to update your view when new information arrives. Directly applicable to how you should think about vol before/after events like earnings.

Before moving on, make sure you've actually run every challenge cell and typed your own explanations in the comments. The act of writing it out is the learning.